In [2]:
import subprocess
subprocess.run(['pip', 'install', 'segmentation-models-pytorch', 'albumentations', 'rasterio', '-q'])
print('Install done')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.4 MB/s eta 0:00:00
Install done


In [3]:
import os, subprocess
if not os.path.exists('/kaggle/working/data/data'):
    subprocess.run(['kaggle','competitions','download','-c','aisehack-theme-1','-p','/kaggle/working'], check=True)
    subprocess.run(['unzip','-q','/kaggle/working/aisehack-theme-1.zip','-d','/kaggle/working/data'], check=True)
    print('Data ready.')
else:
    print('Data already exists.')

100%|██████████| 626M/626M [00:02<00:00, 278MB/s] 



Data ready.


In [4]:
from pathlib import Path
BASE_DIR   = Path('/kaggle/working/data/data')
IMAGE_DIR  = BASE_DIR / 'image'
LABEL_DIR  = BASE_DIR / 'label'
SPLIT_DIR  = BASE_DIR / 'split'
PRED_DIR   = BASE_DIR / 'prediction' / 'image'
OUTPUT_DIR = Path('/kaggle/working')

def read_split(fname):
    with open(SPLIT_DIR / fname) as f:
        return [line.strip() for line in f if line.strip()]

train_ids = read_split('train.txt')
val_ids   = read_split('val.txt')
test_ids  = read_split('pred.txt')
all_ids   = train_ids + val_ids
print(f'Train: {len(train_ids)}, Val: {len(val_ids)}, Test: {len(test_ids)}, All: {len(all_ids)}')

Train: 59, Val: 10, Test: 19, All: 69


In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
TOTAL_CHANNELS = 12

def load_tif(path):
    with rasterio.open(path) as src:
        data = src.read()
    return np.transpose(data.astype(np.float32), (1, 2, 0))

def load_label(path):
    with rasterio.open(path) as src:
        lbl = src.read(1)
    return np.where(lbl == 1, 1, 0).astype(np.uint8)

def normalize_image(img):
    out = np.zeros_like(img, dtype=np.float32)
    for c in range(img.shape[2]):
        ch = img[:, :, c]
        valid = ch[np.isfinite(ch)]
        if len(valid) == 0:
            continue
        p2, p98 = np.percentile(valid, 2), np.percentile(valid, 98)
        if p98 > p2:
            out[:, :, c] = np.clip((ch - p2) / (p98 - p2 + 1e-8), 0, 1)
    return out

def add_derived_channels(img):
    sar_hh = img[:, :, 0]; sar_hv = img[:, :, 1]
    green  = img[:, :, 2]; red    = img[:, :, 3]
    nir    = img[:, :, 4]; swir   = img[:, :, 5]
    ndwi   = (green - nir)  / (green + nir  + 1e-8)
    mndwi  = (green - swir) / (green + swir + 1e-8)
    ndvi   = (nir - red)    / (nir + red    + 1e-8)
    sar_ratio = sar_hh / (sar_hv + 1e-8)
    sar_diff  = sar_hh - sar_hv
    sar_sum   = sar_hh + sar_hv
    def norm01(x):
        mn, mx = x.min(), x.max()
        return np.clip((x - mn) / (mx - mn + 1e-8), 0, 1) if mx > mn else np.zeros_like(x)
    extras = np.stack([
        norm01((ndwi+1)/2), norm01((mndwi+1)/2),
        norm01((ndvi+1)/2), norm01(sar_ratio),
        norm01(sar_diff),   norm01(sar_sum),
    ], axis=2)
    return np.concatenate([img, extras], axis=2)

def compute_iou(pred_mask, true_mask, threshold=0.5):
    pred_bin = (pred_mask > threshold).astype(np.uint8)
    intersection = (pred_bin & true_mask).sum()
    union = (pred_bin | true_mask).sum()
    return 1.0 if union == 0 else intersection / union

print('Utilities ready')

Device: cuda
Utilities ready


In [6]:
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

class FloodDataset(Dataset):
    def __init__(self, ids, image_dir, label_dir=None, transform=None, is_test=False):
        self.ids = ids
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir) if label_dir else None
        self.transform = transform
        self.is_test = is_test
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img = load_tif(self.image_dir / f'{img_id}_image.tif')
        img = normalize_image(img)
        img = add_derived_channels(img)
        if self.is_test:
            if self.transform: img = self.transform(image=img)['image']
            return img, img_id
        lbl = load_label(self.label_dir / f'{img_id}_label.tif')
        if self.transform:
            aug = self.transform(image=img, mask=lbl)
            img, lbl = aug['image'], aug['mask']
        return img, lbl.long()

# EXACT same augmentation as the 0.702 run - zero changes
train_transform = A.Compose([
    A.RandomRotate90(p=0.5),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(translate_percent=0.15, scale=(0.75, 1.25), rotate=(-45, 45), p=0.6),
    A.ElasticTransform(alpha=120, sigma=6, p=0.4),
    A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
    A.GaussNoise(p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.5),
    A.CoarseDropout(num_holes_range=(8,12), hole_height_range=(32,48), hole_width_range=(32,48), p=0.4),
    A.RandomGamma(gamma_limit=(70, 130), p=0.3),
    ToTensorV2(),
])
val_transform = A.Compose([ToTensorV2()])
print('Dataset ready')

Dataset ready


In [7]:
import segmentation_models_pytorch as smp

class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, smooth=1.0, dice_weight=0.6):
        super().__init__()
        self.alpha=alpha; self.gamma=gamma
        self.smooth=smooth; self.dice_weight=dice_weight
    def focal_loss(self, pred, target):
        bce = F.binary_cross_entropy_with_logits(pred, target.float(), reduction='none')
        pt  = torch.exp(-bce)
        alpha_t = self.alpha*target.float()+(1-self.alpha)*(1-target.float())
        return (alpha_t*(1-pt)**self.gamma*bce).mean()
    def dice_loss(self, pred, target):
        pred = torch.sigmoid(pred)
        p = pred.view(-1); t = target.float().view(-1)
        inter = (p*t).sum()
        return 1-(2*inter+self.smooth)/(p.sum()+t.sum()+self.smooth)
    def forward(self, pred, target):
        ps = pred.squeeze(1)
        return (1-self.dice_weight)*self.focal_loss(ps,target)+self.dice_weight*self.dice_loss(ps,target)

print('Loss ready')

Loss ready


In [8]:
from torch.cuda.amp import GradScaler, autocast

# PHASE 1: EXACT same as v3 that scored 0.702 - 100 epochs, no early stopping
print('=== PHASE 1: UNet++ EfficientNet-B5 | 100 epochs (proven formula) ===')

model = smp.UnetPlusPlus(
    encoder_name='efficientnet-b5',
    encoder_weights='imagenet',
    in_channels=TOTAL_CHANNELS,
    classes=1,
    activation=None,
    decoder_attention_type='scse',
).to(DEVICE)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

train_ds     = FloodDataset(train_ids, IMAGE_DIR, LABEL_DIR, transform=train_transform)
val_ds       = FloodDataset(val_ids,   IMAGE_DIR, LABEL_DIR, transform=val_transform)
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

criterion = FocalDiceLoss().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)
scaler    = GradScaler()

EPOCHS    = 100
best_iou  = 0
best_path = OUTPUT_DIR / 'best_phase1.pth'

for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0
    for imgs, masks in train_loader:
        imgs, masks = imgs.float().to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        with autocast():
            preds = model(imgs)
            loss  = criterion(preds, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        train_loss += loss.item()
    model.eval()
    val_iou = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.float().to(DEVICE), masks.to(DEVICE)
            with autocast():
                preds = model(imgs)
            prob = torch.sigmoid(preds).squeeze(1).cpu().numpy()
            gt   = masks.cpu().numpy()
            for p, g in zip(prob, gt):
                val_iou += compute_iou(p, g)
    scheduler.step()
    avg_iou = val_iou / len(val_ds)
    if epoch % 10 == 0 or avg_iou > best_iou:
        print(f'Epoch [{epoch:03d}/{EPOCHS}] Loss: {train_loss/len(train_loader):.4f} | IoU: {avg_iou:.4f}')
    if avg_iou > best_iou:
        best_iou = avg_iou
        torch.save(model.state_dict(), best_path)
        print(f'  >>> Best IoU: {best_iou:.4f} saved!')

print(f'Phase 1 done! Best Val IoU: {best_iou:.4f}')

=== PHASE 1: UNet++ EfficientNet-B5 | 100 epochs (proven formula) ===


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

Model params: 32,044,881
Epoch [001/100] Loss: 0.4105 | IoU: 0.2993
  >>> Best IoU: 0.2993 saved!
Epoch [003/100] Loss: 0.3237 | IoU: 0.3060
  >>> Best IoU: 0.3060 saved!
Epoch [006/100] Loss: 0.2791 | IoU: 0.3081
  >>> Best IoU: 0.3081 saved!
Epoch [007/100] Loss: 0.2565 | IoU: 0.3192
  >>> Best IoU: 0.3192 saved!
Epoch [008/100] Loss: 0.2632 | IoU: 0.3344
  >>> Best IoU: 0.3344 saved!
Epoch [010/100] Loss: 0.2600 | IoU: 0.3582
  >>> Best IoU: 0.3582 saved!
Epoch [011/100] Loss: 0.2569 | IoU: 0.3809
  >>> Best IoU: 0.3809 saved!
Epoch [012/100] Loss: 0.2668 | IoU: 0.3902
  >>> Best IoU: 0.3902 saved!
Epoch [013/100] Loss: 0.2380 | IoU: 0.3931
  >>> Best IoU: 0.3931 saved!
Epoch [015/100] Loss: 0.2587 | IoU: 0.3955
  >>> Best IoU: 0.3955 saved!
Epoch [016/100] Loss: 0.2442 | IoU: 0.4021
  >>> Best IoU: 0.4021 saved!
Epoch [017/100] Loss: 0.2347 | IoU: 0.4072
  >>> Best IoU: 0.4072 saved!
Epoch [018/100] Loss: 0.2405 | IoU: 0.4087
  >>> Best IoU: 0.4087 saved!
Epoch [019/100] Loss: 0.22

In [9]:
# PHASE 2: Fine-tune on ALL 69 images - exact same as v3
print('=== PHASE 2: Fine-tuning on ALL 69 images ===')

model_final = smp.UnetPlusPlus(
    encoder_name='efficientnet-b5',
    encoder_weights='imagenet',
    in_channels=TOTAL_CHANNELS,
    classes=1,
    activation=None,
    decoder_attention_type='scse',
).to(DEVICE)
model_final.load_state_dict(torch.load(best_path))

all_ds     = FloodDataset(all_ids, IMAGE_DIR, LABEL_DIR, transform=train_transform)
all_loader = DataLoader(all_ds, batch_size=4, shuffle=True, num_workers=2, pin_memory=True)

criterion2 = FocalDiceLoss().to(DEVICE)
optimizer2 = torch.optim.AdamW(model_final.parameters(), lr=3e-5, weight_decay=1e-4)
scheduler2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=30)
scaler2    = GradScaler()

EPOCHS2    = 30
final_path = OUTPUT_DIR / 'best_final.pth'
best_loss2 = 999

for epoch in range(1, EPOCHS2+1):
    model_final.train()
    train_loss = 0
    for imgs, masks in all_loader:
        imgs, masks = imgs.float().to(DEVICE), masks.to(DEVICE)
        optimizer2.zero_grad()
        with autocast():
            preds = model_final(imgs)
            loss  = criterion2(preds, masks)
        scaler2.scale(loss).backward()
        scaler2.step(optimizer2); scaler2.update()
        train_loss += loss.item()
    scheduler2.step()
    avg_loss = train_loss / len(all_loader)
    if epoch % 5 == 0:
        print(f'Phase2 [{epoch:02d}/{EPOCHS2}] Loss: {avg_loss:.4f}')
    if avg_loss < best_loss2:
        best_loss2 = avg_loss
        torch.save(model_final.state_dict(), final_path)

print('Phase 2 done!')

=== PHASE 2: Fine-tuning on ALL 69 images ===
Phase2 [05/30] Loss: 0.2182
Phase2 [10/30] Loss: 0.2163
Phase2 [15/30] Loss: 0.2177
Phase2 [20/30] Loss: 0.1989
Phase2 [25/30] Loss: 0.2093
Phase2 [30/30] Loss: 0.2157
Phase 2 done!


In [10]:
# Threshold tuning
model.load_state_dict(torch.load(best_path))
model.eval()

val_ds2     = FloodDataset(val_ids, IMAGE_DIR, LABEL_DIR, transform=val_transform)
val_loader2 = DataLoader(val_ds2, batch_size=4, shuffle=False, num_workers=2)

all_probs, all_gts = [], []
with torch.no_grad():
    for imgs, masks in val_loader2:
        imgs = imgs.float().to(DEVICE)
        with autocast():
            preds = model(imgs)
        all_probs.append(torch.sigmoid(preds).squeeze(1).cpu().numpy())
        all_gts.append(masks.numpy())

all_probs = np.concatenate(all_probs)
all_gts   = np.concatenate(all_gts)

print('Tuning threshold...')
best_t, best_iou_t = 0.5, 0
for t in np.arange(0.10, 0.90, 0.02):
    iou = np.mean([compute_iou(p, g, t) for p, g in zip(all_probs, all_gts)])
    print(f'  t={t:.2f} -> IoU={iou:.4f}')
    if iou > best_iou_t:
        best_iou_t, best_t = iou, t
print(f'Best threshold: {best_t:.2f} -> IoU: {best_iou_t:.4f}')

Tuning threshold...
  t=0.10 -> IoU=0.3805
  t=0.12 -> IoU=0.3901
  t=0.14 -> IoU=0.3987
  t=0.16 -> IoU=0.4061
  t=0.18 -> IoU=0.4126
  t=0.20 -> IoU=0.4181
  t=0.22 -> IoU=0.4228
  t=0.24 -> IoU=0.4266
  t=0.26 -> IoU=0.4296
  t=0.28 -> IoU=0.4319
  t=0.30 -> IoU=0.4339
  t=0.32 -> IoU=0.4356
  t=0.34 -> IoU=0.4373
  t=0.36 -> IoU=0.4384
  t=0.38 -> IoU=0.4394
  t=0.40 -> IoU=0.4401
  t=0.42 -> IoU=0.4406
  t=0.44 -> IoU=0.4408
  t=0.46 -> IoU=0.4409
  t=0.48 -> IoU=0.4406
  t=0.50 -> IoU=0.4402
  t=0.52 -> IoU=0.4398
  t=0.54 -> IoU=0.4392
  t=0.56 -> IoU=0.4384
  t=0.58 -> IoU=0.4374
  t=0.60 -> IoU=0.4360
  t=0.62 -> IoU=0.4341
  t=0.64 -> IoU=0.4321
  t=0.66 -> IoU=0.4294
  t=0.68 -> IoU=0.4263
  t=0.70 -> IoU=0.4224
  t=0.72 -> IoU=0.4181
  t=0.74 -> IoU=0.4128
  t=0.76 -> IoU=0.4069
  t=0.78 -> IoU=0.4003
  t=0.80 -> IoU=0.3928
  t=0.82 -> IoU=0.3839
  t=0.84 -> IoU=0.3732
  t=0.86 -> IoU=0.3605
  t=0.88 -> IoU=0.3455
Best threshold: 0.46 -> IoU: 0.4409


In [11]:
# TTA prediction
model_final.load_state_dict(torch.load(final_path))
model_final.eval()

def predict_tta(img_tensor, model):
    img = img_tensor.unsqueeze(0).float().to(DEVICE)
    def run(x):
        with torch.no_grad():
            with autocast():
                return torch.sigmoid(model(x)).squeeze().cpu().numpy()
    preds = [
        run(img),
        np.fliplr(run(torch.flip(img, [3]))),
        np.flipud(run(torch.flip(img, [2]))),
        np.rot90(run(torch.rot90(img, 1, [2,3])), -1),
        np.rot90(run(torch.rot90(img, 2, [2,3])), -2),
        np.rot90(run(torch.rot90(img, 3, [2,3])), -3),
        np.fliplr(np.flipud(run(torch.flip(img, [2,3])))),
        np.rot90(np.fliplr(run(torch.flip(torch.rot90(img,1,[2,3]),[3]))), -1),
    ]
    return np.mean(preds, axis=0)

def mask_to_rle(mask):
    mask = mask.flatten(order='F')
    pixels = np.concatenate([[0], mask, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(r) for r in runs) if len(runs) else ''

test_ds = FloodDataset(test_ids, PRED_DIR, label_dir=None, transform=val_transform, is_test=True)
rows = []
print('Generating predictions with 8-way TTA...')
for img_t, img_id in test_ds:
    prob = predict_tta(img_t, model_final)
    mask = (prob > best_t).astype(np.uint8)
    rows.append({'id': img_id, 'rle_mask': mask_to_rle(mask)})
    print(f'  {img_id} -> flood pixels: {mask.sum()}')

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print('submission.csv saved!')
print(df.head())

Generating predictions with 8-way TTA...
  20240529_EO4_RES2_fl_pid_080 -> flood pixels: 20754
  20240529_EO4_RES2_fl_pid_081 -> flood pixels: 44013
  20240529_EO4_RES2_fl_pid_082 -> flood pixels: 18829
  20240529_EO4_RES2_fl_pid_083 -> flood pixels: 86830
  20240529_EO4_RES2_fl_pid_084 -> flood pixels: 204218
  20240529_EO4_RES2_fl_pid_085 -> flood pixels: 187714
  20240529_EO4_RES2_fl_pid_086 -> flood pixels: 87249
  20240529_EO4_RES2_fl_pid_087 -> flood pixels: 3746
  20240529_EO4_RES2_fl_pid_088 -> flood pixels: 24107
  20240529_EO4_RES2_fl_pid_089 -> flood pixels: 48649
  20240529_EO4_RES2_fl_pid_090 -> flood pixels: 56218
  20240529_EO4_RES2_fl_pid_091 -> flood pixels: 6166
  20240529_EO4_RES2_fl_pid_092 -> flood pixels: 64091
  20240529_EO4_RES2_fl_pid_093 -> flood pixels: 22669
  20240529_EO4_RES2_fl_pid_094 -> flood pixels: 853
  20240529_EO4_RES2_fl_pid_095 -> flood pixels: 25009
  20240529_EO4_RES2_fl_pid_096 -> flood pixels: 506
  20240529_EO4_RES2_fl_pid_097 -> flood pixel